# Classification NBA Model

## Configuration

## Imports

In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    make_scorer,
    precision_score,
    recall_score,
)
from nba_ou.data_preparation.missing_data.clean_df_for_training import (
    clean_dataframe_for_training
)
from sklearn.model_selection import KFold, cross_validate, train_test_split
from xgboost import XGBClassifier


In [2]:
from nba_ou.modeling.modeling import (
    split_latest_dates_holdout,
    make_walk_forward_last_n_seasons_splits,
    validate_time_splits,
    make_test_anchored_walk_forward_splits,
    assert_valid_time_splits,
    save_model_bundle,
    load_model_bundle
)

In [3]:
nan_threshold = 50.0
max_na_per_row = 70

## Load Data

In [4]:
exclude = "fanatics_sportsbook"

In [5]:
data_path = "/home/adrian_alvarez/Projects/NBA_over_under_predictor/data/train_data/"
name = "all_odds_training_data_until_20260318.csv"

path = data_path + name

df_stats = pd.read_csv(path)

dtype_dict = {col: str for col in df_stats.columns if "ID" in col.upper()}

df_stats = pd.read_csv(
    path,
    dtype=dtype_dict
)
df_stats['GAME_DATE'] = pd.to_datetime(df_stats['GAME_DATE']).dt.strftime('%Y-%m-%d')

In [6]:
df_stats = df_stats[df_stats['SEASON_YEAR'] >= 2021]

In [7]:
df_to_train = clean_dataframe_for_training(df_stats, nan_threshold=nan_threshold, max_na_per_row=max_na_per_row, create_missing_flags=False, verbose=1, keep_columns=['GAME_DATE'], exclude_cols_containing=[exclude])

STARTING DATAFRAME CLEANING PIPELINE
Starting basic cleaning with 6320 rows
Basic cleaning complete: 6305 rows remaining

Starting advanced column cleaning with 2948 columns

Advanced column cleaning complete: 2948 → 2054 columns (894 removed)


Applying missing data policy...

Missing Data Policy Report:
  Rows dropped: 0 (0.0%)
  Critical columns requiring data: 4
  Columns zero-filled: 112
  Infer pairs applied: 0/106
  Remaining NaN cells: 251438

Dropping rows with more than 70 NaN values...
Removed 663 rows exceeding NaN threshold
CLEANING COMPLETE
Final shape: (5642, 2054)


In [8]:
# Count NAs per column
na_counts = df_to_train.isna().sum()

# Get most common SEASON_YEAR for nulls in each column
most_common_season = []
for col in df_to_train.columns:
    if na_counts[col] > 0:
        # Get rows where this column is null
        null_rows = df_to_train[df_to_train[col].isna()]
        if len(null_rows) > 0 and 'SEASON_YEAR' in df_to_train.columns:
            # Find most common SEASON_YEAR for these null rows
            common_season = null_rows['SEASON_YEAR'].mode()
            most_common_season.append(common_season.iloc[0] if len(common_season) > 0 else None)
        else:
            most_common_season.append(None)
    else:
        most_common_season.append(None)

na_counts_df = pd.DataFrame({
    'Column': na_counts.index,
    'NA_Count': na_counts.values,
    'NA_Percentage': (na_counts.values / len(df_to_train) * 100).round(2),
    'Most_Common_Season_Year': most_common_season
}).sort_values('NA_Count', ascending=False)

# Show only columns with NAs
na_counts_df[na_counts_df['NA_Count'] > 0]

,Column,NA_Count,NA_Percentage,Most_Common_Season_Year
1732,total_consensus_pct_under_TREND_SLOPE_LAST_5_H...,761,13.49,2023.0
1730,total_consensus_pct_over_TREND_SLOPE_LAST_5_HO...,748,13.26,2023.0
1736,spread_consensus_pct_home_TREND_SLOPE_LAST_5_H...,678,12.02,2023.0
1734,spread_consensus_pct_away_TREND_SLOPE_LAST_5_H...,667,11.82,2023.0
1731,total_consensus_pct_under_TREND_SLOPE_LAST_5_G...,654,11.59,2023.0
...,...,...,...,...
1918,odds_book_total_prob_diff_novig_betmgm,1,0.02,2021.0
1919,odds_book_total_vig_betmgm,1,0.02,2021.0
1886,spread_betmgm_price_home,1,0.02,2021.0
1874,total_betmgm_price_over,1,0.02,2021.0


In [9]:
BET365_LINE_COL =  "TOTAL_LINE_bet365"
# BET365_LINE_COL =  "total_bet365_line_over"

# Ensure scoring line and target exist (avoid NaN-driven undefined betting accuracy).
df_to_train = df_to_train.dropna(subset=[BET365_LINE_COL, "TOTAL_POINTS"]).copy()

In [10]:
df_to_train['GAME_DATE'] = pd.to_datetime(df_to_train['GAME_DATE'])
df_to_train = df_to_train.sort_values("GAME_DATE").reset_index(drop=True)

In [11]:
#count games per season
games_per_season = df_to_train.groupby('SEASON_YEAR').size()
print(games_per_season)

SEASON_YEAR
2021    1239
2022    1233
2023     967
2024    1238
2025     965
dtype: int64


## Train / Test

In [12]:
from sklearn.model_selection import KFold, cross_validate, cross_val_predict, TimeSeriesSplit
from sklearn.metrics import mean_squared_error, mean_absolute_error, make_scorer, root_mean_squared_error
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from xgboost import XGBRegressor
from sklearn.base import clone

In [13]:
df_dev, df_test_final = split_latest_dates_holdout(
    df=df_to_train,
    date_col="GAME_DATE",
    test_size=0.05,
)

print(f"Development set size: {len(df_dev)}")
print(f"Final test set size: {len(df_test_final)}")
print("Final test date range:",
      df_test_final["GAME_DATE"].min(), "->", df_test_final["GAME_DATE"].max())

Development set size: 5359
Final test set size: 283
Final test date range: 2026-02-04 00:00:00 -> 2026-03-18 00:00:00


In [14]:
TARGET_COL = "TOTAL_POINTS"

EXCLUDE_COLS = [
    "TOTAL_POINTS",
    "SEASON_YEAR",
    "GAME_DATE",
]

X_dev = df_dev.drop(columns=EXCLUDE_COLS, errors="ignore")
y_dev = pd.to_numeric(df_dev[TARGET_COL], errors="coerce")

X_test_final = df_test_final.drop(columns=EXCLUDE_COLS, errors="ignore")
y_test_final = pd.to_numeric(df_test_final[TARGET_COL], errors="coerce")

print(f"X_dev shape: {X_dev.shape}")
print(f"X_test_final shape: {X_test_final.shape}")

X_dev shape: (5359, 2051)
X_test_final shape: (283, 2051)


In [15]:
from nba_ou.modeling.scorers import OverUnderScorerTotalPoints, over_under_betting_accuracy_total_points, evaluate_total_points_thresholds, OverUnderScorerTotalPointsMinEdge
    
ou_scorer = OverUnderScorerTotalPoints(BET365_LINE_COL)

ou_scorer_edge_4 = OverUnderScorerTotalPointsMinEdge(
    line_col=BET365_LINE_COL,
    min_edge=4,
)
ou_scorer_edge_2 = OverUnderScorerTotalPointsMinEdge(
    line_col=BET365_LINE_COL,
    min_edge=2,
)

scoring = {
    "MAE": "neg_mean_absolute_error",
    "RMSE": "neg_root_mean_squared_error",
    "R2": "r2",
    "OU_Betting_Accuracy": ou_scorer,
    "OU_Betting_Accuracy_Edge_4": ou_scorer_edge_4,
    "OU_Betting_Accuracy_Edge_2": ou_scorer_edge_2,
}

def print_metrics(cv_results):
    for sc in scoring.keys():
        train_key = f"train_{sc}"
        test_key = f"test_{sc}"

        train_vals = cv_results[train_key]
        test_vals = cv_results[test_key]

        # Handle NaNs properly (important for edge metrics)
        train_val = np.nanmean(train_vals)
        test_val = np.nanmean(test_vals)

        if sc in {"MSE", "RMSE", "MAE"}:
            train_val = -train_val
            test_val = -test_val

        if "OU_Betting_Accuracy" in sc:
            print(f"Train {sc}: {train_val:.2%}")
            print(f"Validation {sc}: {test_val:.2%}")

            # Optional but VERY useful
            n_valid = np.sum(~np.isnan(test_vals))
            print(f"  (valid folds: {n_valid}/{len(test_vals)})")

        else:
            print(f"Train {sc}: {train_val:.5f}")
            print(f"Validation {sc}: {test_val:.5f}")

        print()

In [16]:
train_games = 2500

splits, fold_info = make_test_anchored_walk_forward_splits(
    df=df_dev,
    date_col="GAME_DATE",
    season_col="SEASON_YEAR",
    test_games=50,
    step_games_between_tests=30,
    train_games=train_games,
    min_train_games=1250,
    max_folds=12,
    verbose=1,
)


Created 12 test-anchored walk-forward folds
 fold  train_n_games  test_n_games train_start_date train_end_date test_start_date test_end_date  test_season
    1           2500            54       2023-01-02     2025-02-23      2025-02-24    2025-03-02         2024
    2           2500            54       2023-01-13     2025-03-06      2025-03-07    2025-03-13         2024
    3           2500            51       2023-01-25     2025-03-17      2025-03-18    2025-03-24         2024
    4           2500            53       2023-02-06     2025-03-29      2025-03-30    2025-04-05         2024
    5           2500            51       2023-02-24     2025-04-09      2025-04-10    2025-04-25         2024
    6           2500            50       2023-03-11     2025-06-22      2025-10-27    2025-11-06         2025
    7           2500            51       2023-03-24     2025-11-10      2025-11-11    2025-11-17         2025
    8           2500            57       2023-04-04     2025-11-22      2025

In [17]:
season_bl = DummyRegressor(strategy="mean")

cv_results = cross_validate(
    season_bl,
    X_dev,
    y_dev,
    cv=splits,
    scoring=scoring,
    return_train_score=True,
    n_jobs=1,
)

print("DummyRegressor baseline")
print_metrics(cv_results)

DummyRegressor baseline
Train MAE: 15.60852
Validation MAE: 15.23402

Train RMSE: 19.50862
Validation RMSE: 18.95043

Train R2: 0.00000
Validation R2: -0.07163

Train OU_Betting_Accuracy: 49.27%
Validation OU_Betting_Accuracy: 50.31%
  (valid folds: 12/12)

Train OU_Betting_Accuracy_Edge_4: 49.99%
Validation OU_Betting_Accuracy_Edge_4: 51.37%
  (valid folds: 12/12)

Train OU_Betting_Accuracy_Edge_2: 50.05%
Validation OU_Betting_Accuracy_Edge_2: 51.79%
  (valid folds: 12/12)



In [18]:
lr = LinearRegression()

cv_results = cross_validate(
    lr,
    X_dev.fillna(0),   # LR cannot handle NaNs
    y_dev,
    cv=splits,
    scoring=scoring,
    return_train_score=True,
    n_jobs=-1,
)

print("Linear Regression")
print_metrics(cv_results)

Linear Regression
Train MAE: 8.29652
Validation MAE: 53.86441

Train RMSE: 10.61655
Validation RMSE: 150.05059

Train R2: 0.70384
Validation R2: -452.77629

Train OU_Betting_Accuracy: 79.33%
Validation OU_Betting_Accuracy: 52.74%
  (valid folds: 12/12)

Train OU_Betting_Accuracy_Edge_4: 85.86%
Validation OU_Betting_Accuracy_Edge_4: 53.14%
  (valid folds: 12/12)

Train OU_Betting_Accuracy_Edge_2: 82.86%
Validation OU_Betting_Accuracy_Edge_2: 52.79%
  (valid folds: 12/12)



In [19]:
xgb_reg = XGBRegressor(
    max_depth=4,
    learning_rate=0.057,
    n_estimators=75,
    subsample=0.8,
    colsample_bytree=0.86,
    reg_alpha=0.57,
    reg_lambda=1.78,
    min_child_weight=5.48,
    gamma=1.77,
    n_jobs=-2,          # choose your CPU count
    random_state=16,
)

cv_results = cross_validate(
    xgb_reg,
    X_dev,
    y_dev,
    cv=splits,
    scoring=scoring,
    return_train_score=True,
    n_jobs=1,          # leave at 1 because XGB already parallelizes internally
)

print("XGBoost")
print_metrics(cv_results)

XGBoost
Train MAE: 10.18014
Validation MAE: 13.93168

Train RMSE: 12.78199
Validation RMSE: 17.26529

Train R2: 0.57072
Validation R2: 0.10712

Train OU_Betting_Accuracy: 83.10%
Validation OU_Betting_Accuracy: 50.53%
  (valid folds: 12/12)

Train OU_Betting_Accuracy_Edge_4: 97.44%
Validation OU_Betting_Accuracy_Edge_4: 55.22%
  (valid folds: 12/12)

Train OU_Betting_Accuracy_Edge_2: 92.67%
Validation OU_Betting_Accuracy_Edge_2: 54.56%
  (valid folds: 12/12)



In [20]:
xgb_reg.fit(X_dev, y_dev)

y_pred_test_total = xgb_reg.predict(X_test_final)

mse = mean_squared_error(y_test_final, y_pred_test_total)
rmse = root_mean_squared_error(y_test_final, y_pred_test_total)
mae = mean_absolute_error(y_test_final, y_pred_test_total)

betting_line = X_test_final[BET365_LINE_COL].to_numpy(dtype=float)

ou_acc = over_under_betting_accuracy_total_points(
    y_test_final,
    y_pred_test_total,
    betting_line,
)

print("Final test metrics")
print(f"MSE: {mse:.5f}")
print(f"RMSE: {rmse:.5f}")
print(f"MAE: {mae:.5f}")
print(f"OU_Betting_Accuracy: {ou_acc:.2%}")

Final test metrics
MSE: 289.98938
RMSE: 17.02907
MAE: 13.49838
OU_Betting_Accuracy: 54.87%


In [21]:
results_df, y_pred_test_total = evaluate_total_points_thresholds(
    model=xgb_reg,
    X_test=X_test_final,
    y_test_total=y_test_final,
    line_col=BET365_LINE_COL,
    thresholds=range(0, 11),
)

display(
    results_df.style.format(
        {"pct_of_test": "{:.1%}", "ou_betting_accuracy": "{:.2%}"}
    )
)

,threshold_abs_pred_edge_gt,n_games,pct_of_test,ou_betting_accuracy
0,0,283,100.0%,54.87%
1,1,194,68.6%,54.50%
2,2,117,41.3%,57.14%
3,3,65,23.0%,58.06%
4,4,30,10.6%,60.00%
5,5,11,3.9%,81.82%
6,6,6,2.1%,83.33%
7,7,3,1.1%,66.67%
8,8,2,0.7%,50.00%
9,9,1,0.4%,100.00%


# OPTUNA

In [22]:
from nba_ou.modeling.optuna_total_points import (
    tune_xgb_total_points_optuna,
    summarize_optuna_trials,
    fit_best_xgb_total_points,
)

study = tune_xgb_total_points_optuna(
    X=X_dev,
    y=y_dev,
    splits=splits,
    line_col=BET365_LINE_COL,
    n_trials=80,
    timeout=4.5 * 3600,
    objective_name="reg:squarederror",   # second run: reg:pseudohubererror
    study_name="xgb_total_points_mae",
)

print("Best CV MAE:", study.best_value)
print("Best params:")
for k, v in study.best_params.items():
    print(f"{k}: {v}")

print("\nSecondary metrics from best trial:")
print("Mean RMSE:", study.best_trial.user_attrs.get("mean_rmse"))
print("Mean R2:", study.best_trial.user_attrs.get("mean_r2"))
print("Mean OU accuracy:", study.best_trial.user_attrs.get("mean_ou_acc"))
print("Mean best_iteration:", study.best_trial.user_attrs.get("mean_best_iteration"))

trials_df = summarize_optuna_trials(study)
display(
    trials_df.head(15).style.format(
        {
            "value_mae": "{:.4f}",
            "mean_rmse": "{:.4f}",
            "mean_r2": "{:.4f}",
            "mean_ou_acc": "{:.2%}",
        }
    )
)

[I 2026-03-20 20:13:14,280] A new study created in memory with name: xgb_total_points_mae


  0%|          | 0/80 [00:00<?, ?it/s]

[I 2026-03-20 20:20:12,638] Trial 0 finished with value: 13.592072471171285 and parameters: {'max_depth': 2, 'min_child_weight': 18.346704707583235, 'gamma': 1.6970342240854253, 'subsample': 0.5682407800531226, 'colsample_bytree': 0.5123279759064977, 'learning_rate': 0.011926786034588454, 'reg_alpha': 1.8771791376898666, 'reg_lambda': 1.897469395521307}. Best is trial 0 with value: 13.592072471171285.
[I 2026-03-20 20:25:38,127] Trial 1 finished with value: 13.667507783897129 and parameters: {'max_depth': 2, 'min_child_weight': 51.819268381786635, 'gamma': 1.7346760026809933, 'subsample': 0.5811969357559948, 'colsample_bytree': 0.675188230006178, 'learning_rate': 0.010426963054371881, 'reg_alpha': 0.06701717253228541, 'reg_lambda': 3.152289132591451}. Best is trial 0 with value: 13.592072471171285.
[I 2026-03-20 20:29:30,881] Trial 2 finished with value: 13.555061285036963 and parameters: {'max_depth': 4, 'min_child_weight': 15.848753157115912, 'gamma': 0.7236802167715846, 'subsample':

,trial,value_mae,mean_rmse,mean_r2,mean_ou_acc,mean_ou_acc_edge_2,mean_ou_acc_edge_3,mean_ou_acc_edge_4,mean_best_iteration,median_best_iteration,max_depth,min_child_weight,gamma,subsample,colsample_bytree,learning_rate,reg_alpha,reg_lambda
0,35,13.4538,16.7926,0.1565,56.27%,0.635226,0.620670,0.549934,133,79,2,47.321584,1.781008,0.858954,0.690601,0.042562,0.480496,15.066708
1,37,13.4616,16.8861,0.1480,57.12%,0.612572,0.552155,0.584576,144,76,2,44.295515,0.896526,0.784061,0.616215,0.049447,1.322696,15.581883
2,43,13.4651,16.8424,0.1519,55.41%,0.639566,0.602355,0.532107,132,62,2,48.272909,0.511523,0.894259,0.640888,0.043299,0.505217,18.535861
3,16,13.4682,16.8737,0.1488,56.02%,0.606842,0.609444,0.689516,168,112,2,11.246665,2.189172,0.863247,0.605022,0.030899,0.010189,27.431807
4,56,13.4788,16.8499,0.1513,56.05%,0.618495,0.606755,0.651001,126,69,2,47.331403,0.981765,0.823820,0.702258,0.039973,1.120243,18.223925
5,45,13.4821,16.8860,0.1477,55.55%,0.624000,0.569050,0.583958,109,55,2,34.221109,0.482960,0.791801,0.705644,0.045764,2.500574,18.579363
6,34,13.4825,16.8293,0.1533,57.92%,0.612097,0.584123,0.628862,138,85,2,56.614315,1.755427,0.857108,0.605659,0.042606,0.338829,14.836765
7,32,13.4840,16.8653,0.1497,55.73%,0.619459,0.608774,0.548096,145,74,2,10.858227,1.950080,0.852294,0.526683,0.043000,0.095645,12.763239
8,21,13.4866,16.8918,0.1475,55.63%,0.599932,0.651977,0.519529,133,75,2,9.955134,2.081256,0.843390,0.688300,0.031598,0.010254,12.764571
9,24,13.4877,16.8764,0.1487,56.79%,0.614123,0.595402,0.563638,120,90,2,7.181129,2.040170,0.843747,0.550328,0.033283,10.436529,4.831998


In [23]:
total_df = df_dev.tail(train_games)


In [24]:
X_dev = total_df.drop(columns=EXCLUDE_COLS, errors="ignore")
y_dev = pd.to_numeric(total_df[TARGET_COL], errors="coerce")

In [25]:
best_model = fit_best_xgb_total_points(
    X_dev=X_dev,
    y_dev=y_dev,
    study=study,
    objective_name="reg:squarederror",
)

y_pred_test_total = best_model.predict(X_test_final)

mse = mean_squared_error(y_test_final, y_pred_test_total)
rmse = root_mean_squared_error(y_test_final, y_pred_test_total)
mae = mean_absolute_error(y_test_final, y_pred_test_total)

betting_line = X_test_final[BET365_LINE_COL].to_numpy(dtype=float)

ou_acc = over_under_betting_accuracy_total_points(
    y_true=y_test_final,
    y_pred=y_pred_test_total,
    betting_line=betting_line,
)

print("Final test metrics")
print(f"MSE: {mse:.5f}")
print(f"RMSE: {rmse:.5f}")
print(f"MAE: {mae:.5f}")
print(f"OU_Betting_Accuracy: {ou_acc:.2%}")

Final test metrics
MSE: 295.26151
RMSE: 17.18317
MAE: 13.55658
OU_Betting_Accuracy: 52.35%


In [26]:
# Limit to Season year 2023 onwards
train_data = df_to_train.tail(train_games).copy()

In [27]:
from nba_ou.modeling.modeling import ModelBundleMetadata, ModelInfo, TrainingMetrics

X_full = train_data.drop(columns=EXCLUDE_COLS, errors="ignore")
y_full = pd.to_numeric(train_data[TARGET_COL], errors="coerce")


production_model = fit_best_xgb_total_points(
    X_dev=X_full,
    y_dev=y_full,
    study=study,
    objective_name="reg:squarederror",
)

latest_training_date = pd.to_datetime(train_data["GAME_DATE"]).max()
model_version = latest_training_date.strftime("%d_%m_%y")
model_name = f"three_seasons_xgb_total_points_{model_version}"

metadata = ModelBundleMetadata(
    model_info=ModelInfo(
        name=model_name,
        model_version=model_version,
        model_type="three_seasons_total_points",
        prediction_source="three_seasons_xgb_total_points",
        training_code_tag="1.0",
    ),
    training_metrics=TrainingMetrics(
        best_params=study.best_trial.params,
        mean_best_iteration=study.best_trial.user_attrs.get("mean_best_iteration"),
        cv_mae=float(study.best_value),
        cv_rmse=study.best_trial.user_attrs.get("mean_rmse"),
        cv_ou_acc=study.best_trial.user_attrs.get("mean_ou_acc"),
        final_test_mae=float(mae),
        final_test_rmse=float(rmse),
        final_test_ou_acc=float(ou_acc),
        nan_threshold=nan_threshold,
        max_na_per_row=max_na_per_row,
        train_date_min=train_data["GAME_DATE"].min().to_pydatetime(),
        train_date_max=train_data["GAME_DATE"].max().to_pydatetime(),
    ),
)

model_path, meta_path = save_model_bundle(
    model=production_model,
    feature_names=list(X_full.columns),
    out_dir="/home/adrian_alvarez/Projects/NBA_over_under_predictor/models/total_points/3_seasons/",
    metadata=metadata,
)

print(f"Production model trained on {len(X_full)} rows using fixed n_estimators from mean_best_iteration.")
print("Saved model :", model_path)
print("Saved metadata:", meta_path)


Production model trained on 2500 rows using fixed n_estimators from mean_best_iteration.
Saved model : /home/adrian_alvarez/Projects/NBA_over_under_predictor/models/total_points/3_seasons/three_seasons_xgb_total_points_18_03_26.json
Saved metadata: /home/adrian_alvarez/Projects/NBA_over_under_predictor/models/total_points/3_seasons/three_seasons_xgb_total_points_18_03_26.meta.json
